In [1]:
# Cell 1: Install dependencies
!pip install -q streamlit
!pip install -q transformers==4.45.0 accelerate==0.27.2
!pip install -q scipy pillow opencv-python
!pip install -q nltk rouge-score

print("All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 85.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00:00:01
All libraries installed!


In [2]:
# Cell 2: Verify GPU
import torch

print("=" * 60)
print("SYSTEM CHECK")
print("=" * 60)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("NO GPU! Go back to Settings and enable GPU T4")
    
print("=" * 60)


SYSTEM CHECK
✓ PyTorch version: 2.9.0+cu126
✓ CUDA available: True
✓ GPU: Tesla P100-PCIE-16GB
✓ GPU Memory: 15.9 GB


In [3]:
# Cell 3: Setup - Copy files to working directory
import sys
import os
from pathlib import Path
import shutil

print("Setting up project files...\n")

# Source path (read-only dataset)
DATASET_PATH = "/kaggle/input/datasets/ravinduwellalage2/xai-chest-x-ray-project-final-year/xai-project/kaggle-upload"

# Working directory (writable)
WORK_DIR = "/kaggle/working"

# Copy src_v2 to working directory
src_source = Path(f"{DATASET_PATH}/src_v2")
src_dest = Path(f"{WORK_DIR}/src_v2")

print(f"Copying src_v2 folder to working directory...")

# Remove if exists (in case you're re-running)
if src_dest.exists():
    shutil.rmtree(src_dest)

# Copy the entire folder
shutil.copytree(src_source, src_dest)

# Add to Python path
sys.path.insert(0, WORK_DIR)

print(f"Code copied to: {src_dest}")
print(f"   Files copied: {len(list(src_dest.glob('*.py')))} Python files")

# Fix relative imports in the copied files
print(f"\n🔧 Fixing relative imports...")

for py_file in src_dest.glob("*.py"):
    content = py_file.read_text()
    # Replace relative imports with absolute imports
    content = content.replace("from .config import", "from src_v2.config import")
    content = content.replace("from .xai_enhanced import", "from src_v2.xai_enhanced import")
    content = content.replace("from .behavior_extractor import", "from src_v2.behavior_extractor import")
    content = content.replace("from .llm_explainer import", "from src_v2.llm_explainer import")
    content = content.replace("from .prompt_utils import", "from src_v2.prompt_utils import")
    content = content.replace("from .visualization import", "from src_v2.visualization import")
    
    # Write back
    py_file.write_text(content)

print(f"Imports fixed!")

# Verify other files
model_path = Path(f"{DATASET_PATH}/models/denseNet121_v2.pth")
sample_path = Path(f"{DATASET_PATH}/Test-Samples/sample-2.jpg")

if model_path.exists():
    print(f"\nModel found: {model_path.name}")
    print(f"   Size: {model_path.stat().st_size / 1024**2:.1f} MB")
else:
    print(f"\nModel NOT found at {model_path}")

if sample_path.exists():
    print(f"\nTest image found: {sample_path.name}")
else:
    print(f"\nTest image NOT found at {sample_path}")

# Store paths for later use
MODEL_PATH = str(model_path)
SAMPLE_PATH = str(sample_path)

print("\n" + "=" * 60)
print("Setup complete! Ready to import modules.")
print("=" * 60)


Setting up project files...

Copying src_v2 folder to working directory...
Code copied to: /kaggle/working/src_v2
   Files copied: 9 Python files

🔧 Fixing relative imports...
Imports fixed!

Model found: denseNet121_v2.pth
   Size: 80.6 MB

Test image found: sample-2.jpg

Setup complete! Ready to import modules.


In [4]:
# Cell 5: Import your modules
print("Loading your code modules...\n")

from src_v2.config import *
from src_v2.xai_enhanced import EnhancedGradCAM
from src_v2.behavior_extractor import ModelBehaviorExtractor

print("Config loaded")
print("Grad-CAM module loaded")
print("Behavior Extractor loaded")

# We'll load LLM separately in next cell
print("\nCore modules ready!")


Loading your code modules...

Using device: cuda
Configuration loaded successfully!
Config loaded
Grad-CAM module loaded
Behavior Extractor loaded

Core modules ready!


In [5]:
# Cell 6.5: Fix gaussian_filter compatibility issue
import sys

# Read the xai_enhanced.py file
xai_file = Path("/kaggle/working/src_v2/xai_enhanced.py")
content = xai_file.read_text()

# Find and replace the gaussian_filter section
old_code = """        # ADD SMOOTHING HERE
        from scipy.ndimage import gaussian_filter
        cam = gaussian_filter(cam, sigma=2)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # Re-normalize"""

new_code = """        # ADD SMOOTHING HERE (with dtype fix for scipy)
        from scipy.ndimage import gaussian_filter
        # Convert to float32 for scipy compatibility
        cam = cam.astype(np.float32)
        cam = gaussian_filter(cam, sigma=2)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # Re-normalize"""  

content = content.replace(old_code, new_code)

# Write back
xai_file.write_text(content)

print("Fixed gaussian_filter compatibility issue!")
print("Now supports GPU float16 operations")


Fixed gaussian_filter compatibility issue!
Now supports GPU float16 operations


In [6]:
# Cell 6.6: Reload the fixed module
import importlib
import sys

# Remove cached module
if 'src_v2.xai_enhanced' in sys.modules:
    del sys.modules['src_v2.xai_enhanced']

# Re-import
from src_v2.xai_enhanced import EnhancedGradCAM

print("Module reloaded with fix!")


Module reloaded with fix!


In [7]:
import torch.nn as nn
from torchvision import models

print("Rebuilding your DenseNet-121 architecture...")

class ChestXRayModel(nn.Module):
    def __init__(self, num_classes=14):
        super(ChestXRayModel, self).__init__()
        self.backbone = models.densenet121(weights=None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = ChestXRayModel(num_classes=14)
checkpoint = torch.load(MODEL_PATH, map_location='cuda', weights_only=False)

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
    print(f"  Checkpoint from epoch {checkpoint.get('epoch', '?')}")
else:
    state_dict = checkpoint

state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=True)  # strict=True confirms perfect match
model.eval()
model.to('cuda')
print(f"DenseNet-121 is ready!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("Done!")

Rebuilding your DenseNet-121 architecture...
  Checkpoint from epoch 15
DenseNet-121 is ready!
   Parameters: 7.0M
Done!


In [8]:
# ═══════════════════════════════════════════════════════════════
# [COMPONENT 3: NLP] — Load Qwen2.5-3B-Instruct
# ═══════════════════════════════════════════════════════════════
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

QWEN_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("=" * 70)
print("COMPONENT 3: NLP — LOADING QWEN2.5-3B-INSTRUCT")
print("=" * 70)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)
print("Tokenizer ready")

print("Loading model in float16...")
llm_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
llm_model.eval()

print(f"\nQwen2.5-3B loaded!")
print(f"   GPU Memory used : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("=" * 70)


COMPONENT 3: NLP — LOADING QWEN2.5-3B-INSTRUCT

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Qwen2.5-3B loaded!
   GPU Memory used : 5.96 GB


# Gradio UI implementation 

In [9]:
import gradio as gr
import torch, numpy as np, cv2, threading, sys
from PIL import Image
from transformers import TextIteratorStreamer


import nest_asyncio
nest_asyncio.apply()

try:
    demo.close()
except:
    pass
# ── Your existing notebook variables ──────────────
# model     = DenseNet-121  (from model loading cell)
# llmmodel  = Qwen2.5-3B    (from Qwen loading cell)
# tokenizer = Qwen tokenizer (from Qwen loading cell)
# ──────────────────────────────────────────────────

LABEL_NAMES = [
    "Atelectasis","Cardiomegaly","Consolidation","Edema",
    "Effusion","Emphysema","Fibrosis","Hernia",
    "Infiltration","Mass","Nodule","Pleural_Thickening",
    "Pneumonia","Pneumothorax"
]

# ─── CV Pipeline ──────────────────────────────────
def run_cv(pil_img):
    from torchvision import transforms

    # ── Input Validation: Reject non-X-ray images ──────────
    img_array = np.array(pil_img.convert("RGB"))
    
    # Check 1: X-rays are nearly grayscale (R≈G≈B channels)
    r, g, b = img_array[:,:,0], img_array[:,:,1], img_array[:,:,2]
    rg_diff = float(np.mean(np.abs(r.astype(int) - g.astype(int))))
    rb_diff = float(np.mean(np.abs(r.astype(int) - b.astype(int))))
    is_grayscale = (rg_diff < 15) and (rb_diff < 15)  # X-rays are nearly grayscale
    
    # Check 2: X-rays have high contrast (dark lungs + bright bones)
    gray = np.array(pil_img.convert("L"))
    std_dev = float(np.std(gray))
    has_contrast = std_dev > 30  # Very flat images or natural photos fail this

    # Check 3: X-rays have significant dark regions (lung fields are dark)
    dark_pixel_ratio = float(np.mean(gray < 80))
    has_dark_regions = dark_pixel_ratio > 0.15  # At least 15% dark pixels

    if not is_grayscale:
        raise gr.Error(
            "Invalid input: This appears to be a colour photograph, not a chest X-ray. "
            "Please upload a valid grayscale chest X-ray image."
        )
    
    if not has_contrast or not has_dark_regions:
        raise gr.Error(
            "Invalid input: This image does not resemble a chest X-ray. "
            "Please upload a valid chest X-ray image."
        )
    # ── End Validation ──────────────────────────────────────

    tfm = transforms.Compose([
        transforms.Resize((224,224)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    t = tfm(pil_img.convert("RGB")).unsqueeze(0).to("cuda")
    with torch.no_grad():
        probs = torch.sigmoid(model(t)).cpu().numpy()[0]
    top_ids = np.argsort(probs)[::-1][:5]
    return t, probs, top_ids, LABEL_NAMES[top_ids[0]], float(probs[top_ids[0]])





# ─── NEW: Lung Region Map Generator ──────────────
def generate_region_map(orig_pil, heatmap_7x7):
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    import matplotlib.lines as mlines
    import io

    arr   = np.array(orig_pil.convert("RGB").resize((224,224)))
    h_res = cv2.resize(heatmap_7x7, (224,224))
    h_col = cv2.cvtColor(
        cv2.applyColorMap((h_res*255).astype(np.uint8), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB
    )
    overlay = cv2.addWeighted(arr, 0.55, h_col, 0.45, 0)

    # Compute 7-region scores from raw 7x7 heatmap
    h = heatmap_7x7
    scores = {
        "Upper-L": float(h[0:3, 0:3].mean()),
        "Upper-R": float(h[0:3, 4:7].mean()),
        "Mid-L":   float(h[2:5, 0:3].mean()),
        "Mid-R":   float(h[2:5, 4:7].mean()),
        "Lower-L": float(h[4:7, 0:3].mean()),
        "Lower-R": float(h[4:7, 4:7].mean()),
        "Central": float(h[2:5, 2:5].mean()),
    }

    def attn_hex(s):
        return "#FF6B6B" if s>0.7 else "#FFB347" if s>0.3 else "#C9B1FF"
    def attn_rgba(s):
        if s>0.7: return (1.0, 0.42, 0.42, 0.22)
        if s>0.3: return (1.0, 0.70, 0.28, 0.22)
        return (0.79, 0.69, 1.0, 0.22)

    fig, ax = plt.subplots(figsize=(6,6))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#0d1117")
    ax.imshow(overlay)
    ax.axis("off")

    grid = [
        (0,   0,   112, 75, "Upper-L", "Upper-L"),
        (112, 0,   112, 75, "Upper-R", "Upper-R"),
        (0,   75,  112, 75, "Mid-L",   "Mid-L"),
        (112, 75,  112, 75, "Mid-R",   "Mid-R"),
        (0,   150, 112, 74, "Lower-L", "Lower-L"),
        (112, 150, 112, 74, "Lower-R", "Lower-R"),
    ]
    for gx, gy, gw, gh, lbl, key in grid:
        s = scores.get(key, 0)
        ax.add_patch(patches.Rectangle((gx,gy), gw, gh,
            facecolor=attn_rgba(s), linewidth=0))
        ax.add_patch(patches.Rectangle((gx,gy), gw, gh,
            facecolor="none", linewidth=1.5,
            edgecolor=attn_hex(s), linestyle="--"))
        ax.text(gx+gw/2, gy+12, lbl, ha="center", va="center",
            color="white", fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="black", alpha=0.65, linewidth=0))
        ax.text(gx+gw/2, gy+gh/2, f"{s*100:.0f}%", ha="center", va="center",
            color="white", fontsize=14, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor=attn_hex(s), alpha=0.82, edgecolor="none"))

    c = scores.get("Central", 0)
    ax.add_patch(patches.Rectangle((50,196), 124, 20,
        facecolor="black", alpha=0.72, linewidth=1.5, edgecolor=attn_hex(c)))
    ax.text(112, 206, f"CENTRAL ZONE  {c*100:.0f}%", ha="center", va="center",
        color="white", fontsize=8.5, fontweight="bold")

    h1 = mlines.Line2D([],[],color="#FF6B6B",marker="o",linestyle="None",markersize=9,label="High (>70%)")
    h2 = mlines.Line2D([],[],color="#FFB347",marker="o",linestyle="None",markersize=9,label="Moderate (30-70%)")
    h3 = mlines.Line2D([],[],color="#C9B1FF",marker="o",linestyle="None",markersize=9,label="Low (<30%)")
    ax.legend(handles=[h1,h2,h3], loc="lower right", fontsize=8.5,
        facecolor="#0d1117", edgecolor="#58a6ff", labelcolor="white", framealpha=0.88)

    plt.tight_layout(pad=0)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight", facecolor="#0d1117")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).copy()


# ─── XAI ──────────────────────────────────────────
def run_xai(img_tensor, disease_idx, orig_pil):
    from src_v2.xai_enhanced import EnhancedGradCAM
    from src_v2.behavior_extractor import ModelBehaviorExtractor
    gradcam   = EnhancedGradCAM(model, target_layer="backbone.features.denseblock4")
    heatmap   = gradcam.generate_cam(img_tensor, disease_idx, device="cuda")  # 7x7
    extractor = ModelBehaviorExtractor()
    behavior  = extractor.extract_complete_behavior(
        predictions={LABEL_NAMES[disease_idx]: 1.0},
        heatmap=heatmap, all_probabilities=np.zeros(14)
    )
    region_map = generate_region_map(orig_pil, heatmap)   # ← NEW
    return behavior, region_map


# ─── Streaming Explanation ────────────────────────
def gen_explanation(disease, confidence, behavior, role):
    spatial = behavior.get("spatial_analysis", {})
    regions = behavior.get("anatomical_regions", [])
    reg_str = ", ".join(
        f"{r['name'].replace('_',' ')} ({r['attention_score']*100:.0f}%)"
        for r in regions[:3]
    ) or "unspecified regions"
    pattern = "focal" if spatial.get("is_focal") else "diffuse"
    cov     = spatial.get("attention_percentage", 0)

    if role == "Clinician":
        user_msg = (
            f"Generate a structured radiological report.\n"
            f"Pathology: {disease} | Confidence: {confidence*100:.1f}%\n"
            f"Pattern: {pattern} | Coverage: {cov:.1f}% | Zones: {reg_str}\n\n"
            f"Sections: 1.CLINICAL IMPRESSION  2.FINDINGS  "
            f"3.GRAD-CAM ANALYSIS  4.RECOMMENDATION\n"
            f"Use precise radiological terminology."
        )
    else:
        user_msg = (
            f"Explain this chest X-ray to a patient in warm simple language.\n"
            f"Finding: {disease} | AI focus zones: {reg_str}\n\n"
            f"Write 5-6 warm, reassuring sentences. Use 'you'/'your'. "
            f"No medical jargon. End encouragingly."
        )

    messages = [
        {"role":"system","content":"You are CheXplain, an expert medical AI."},
        {"role":"user","content":user_msg}
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True, timeout=120)
    gen_kw   = dict(**inputs, max_new_tokens=650, temperature=0.7, top_p=0.9,
                    do_sample=True, repetition_penalty=1.1,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.eos_token_id, streamer=streamer)
    threading.Thread(target=llm_model.generate, kwargs=gen_kw, daemon=True).start()
    full = ""
    for tok in streamer:
        full += tok
        yield full

# ─── Chat Response ────────────────────────────────
def get_chat_resp(msg, history, role, data):
    regions = data.get("anatomical_regions",[])
    spatial = data.get("spatial_analysis",{})
    reg_str = ", ".join(
        f"{r['name'].replace('_',' ')} ({r['attention_score']*100:.0f}%)"
        for r in regions[:3]
    ) or "—"
    pattern = "Focal" if spatial.get("is_focal") else "Diffuse"
    sys_c = (
        f"You are CheXplain, specialist AI for THIS chest X-ray.\n"
        f"Finding: {data.get('disease','?')} | "
        f"{data.get('confidence',0)*100:.1f}% | {pattern} | {reg_str}\n"
        f"{'Use clinical precision.' if role=='Clinician' else 'Use simple warm language.'}\n\n"
        f"FULL {'REPORT' if role=='Clinician' else 'EXPLANATION'}:\n"
        f"{data.get('clinician_report' if role=='Clinician' else 'patient_report','')}"
    )
    messages = [{"role":"system","content":sys_c}]
    for m in history[1:][-4:]:
        messages.append({"role": m["role"], "content": m["content"]})
    messages.append({"role":"user","content":msg})
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt", truncation=True, max_length=3072).to("cuda")
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=500, temperature=0.7,
            top_p=0.9, do_sample=True, repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# ════════════════════════════════════════════════════
# GRADIO BLOCKS UI
# ════════════════════════════════════════════════════
CSS = """
body, .gradio-container{background:#0d1117 !important; color:#e6edf3}
#title{text-align:center;padding:8px 0 16px;border-bottom:1px solid #21262d;margin-bottom:14px}
#title h1{font-size:24px;color:#58a6ff;margin:0}
#title p{font-size:12px;color:#8b949e;margin:4px 0 0}
.badge{background:#161b22;border:1px solid #21262d;border-radius:8px;padding:10px 14px}
footer{display:none !important}
"""

with gr.Blocks(css=CSS, title="CheXplain") as demo:

    # State
    state      = gr.State({})
    cli_hist   = gr.State([])
    pat_hist   = gr.State([])

    # Header
    gr.HTML("""<div id="title">
      <h1>🫁 CheXplain</h1>
      <p>Explainable AI · DenseNet-121 + Grad-CAM + Qwen2.5-3B</p>
    </div>""")

    with gr.Row():
        # ── LEFT PANEL ─────────────────────────────
        with gr.Column(scale=1):
            upload   = gr.Image(type="pil", label="📤 Upload Chest X-Ray", height=210)
            with gr.Row():
                orig_img = gr.Image(label="🩻 X-Ray",   interactive=False, height=210)
                heat_img = gr.Image(label="🫁 Lung Region Map", interactive=False, height=210)
            badge_html  = gr.HTML("")
            status_html = gr.HTML("")
            del_btn     = gr.Button("🗑 Delete / Upload New", variant="secondary", visible=False)

        # ── RIGHT PANEL ────────────────────────────
        with gr.Column(scale=1):
            role = gr.Radio(["Clinician","Patient"], value="Clinician", label="")

            with gr.Group(visible=True) as explain_grp:
                expl_box = gr.Textbox(
                    label="🔬 AI Explanation", lines=12, max_lines=18,
                    interactive=False,
                    placeholder="Upload a chest X-ray to begin…"
                )
                chat_btn = gr.Button("💬 Start Chat", variant="primary", visible=False)

            with gr.Group(visible=False) as chat_grp:
                chatbot = gr.Chatbot(label="", height=360, type="messages", allow_tags=False)
                with gr.Row():
                    chat_in  = gr.Textbox(placeholder="Ask about this X-ray…",
                                          label="", scale=5, container=False)
                    send_btn = gr.Button("Send ➤", variant="primary", scale=1)
                with gr.Row():
                    back_btn  = gr.Button("← Back to Explanation", scale=1)
                    clear_btn = gr.Button("🗑 Clear Chat", scale=1)

    # ── UPLOAD → PIPELINE ──────────────────────────
    def pipeline(pil_img, sel_role):
        if pil_img is None:
            yield (None, None,
                   gr.update(value=""), gr.update(value=""),
                   gr.update(visible=False), gr.update(value=""),
                   gr.update(visible=False), gr.update(visible=True),
                   gr.update(visible=False), {}, [], [])
            return

        yield (pil_img, None, gr.update(value=""),
               gr.update(value='<p style="color:#58a6ff">⟳ Running DenseNet-121…</p>'),
               gr.update(visible=False), gr.update(value=""),
               gr.update(visible=False), gr.update(visible=True),
               gr.update(visible=False), {}, [], [])

        img_tensor, probs, top_ids, disease, conf = run_cv(pil_img)

        yield (pil_img, None, gr.update(value=""),
               gr.update(value=f'<p style="color:#58a6ff">✓ {disease} ({conf*100:.1f}%) — Generating heatmap…</p>'),
               gr.update(visible=False), gr.update(value=""),
               gr.update(visible=False), gr.update(visible=True),
               gr.update(visible=False), {}, [], [])

        behavior, overlay = run_xai(img_tensor, top_ids[0], pil_img)
        sp      = behavior.get("spatial_analysis", {})
        regions = behavior.get("anatomical_regions", [])
        pattern = "Focal" if sp.get("is_focal") else "Diffuse"
        zones   = " · ".join(r["name"].replace("_"," ") for r in regions[:3]) or "—"

        bdg = f"""<div class="badge">
          <span style="background:#da3633;color:#fff;font-size:12px;font-weight:700;
                       padding:3px 10px;border-radius:16px">⚠ {disease}</span>
          <span style="background:#238636;color:#fff;font-size:12px;font-weight:700;
                       padding:3px 10px;border-radius:16px;margin-left:6px">
            ✓ {conf*100:.1f}%</span>
          <div style="font-size:12px;color:#8b949e;margin-top:6px">
            Pattern: <b style="color:#cdd9e5">{pattern}</b> &nbsp;·&nbsp;
            Zones: <b style="color:#cdd9e5">{zones}</b>
          </div></div>"""

        yield (pil_img, overlay, gr.update(value=bdg),
               gr.update(value=f'<p style="color:#58a6ff">✓ Heatmap ready — Generating {sel_role} explanation…</p>'),
               gr.update(visible=False), gr.update(value="Generating…"),
               gr.update(visible=False), gr.update(visible=True),
               gr.update(visible=False), {}, [], [])

        # Stream selected role explanation
        current = ""
        for partial in gen_explanation(disease, conf, behavior, sel_role):
            current = partial
            yield (pil_img, overlay, gr.update(value=bdg),
                   gr.update(value=f'<p style="color:#58a6ff">⟳ Streaming explanation…</p>'),
                   gr.update(visible=False), gr.update(value=current+"▌"),
                   gr.update(visible=False), gr.update(visible=True),
                   gr.update(visible=False), {}, [], [])

        cli_rep = current if sel_role=="Clinician" else ""
        pat_rep = current if sel_role=="Patient"   else ""

        # Generate other role silently
        yield (pil_img, overlay, gr.update(value=bdg),
               gr.update(value='<p style="color:#58a6ff">⟳ Preparing both roles…</p>'),
               gr.update(visible=False), gr.update(value=current),
               gr.update(visible=False), gr.update(visible=True),
               gr.update(visible=False), {}, [], [])

        other_role = "Patient" if sel_role=="Clinician" else "Clinician"
        other = ""
        for partial in gen_explanation(disease, conf, behavior, other_role):
            other = partial
        if sel_role=="Clinician": pat_rep = other
        else:                     cli_rep = other

        final = {
            "disease": disease, "confidence": float(conf),
            "spatial_analysis": {
                "is_focal": bool(sp.get("is_focal",False)),
                "attention_percentage": float(sp.get("attention_percentage",0))
            },
            "anatomical_regions": [
                {"name":r["name"],"attention_score":float(r["attention_score"])}
                for r in regions[:5]
            ],
            "clinician_report": cli_rep,
            "patient_report":   pat_rep,
        }

        yield (pil_img, overlay, gr.update(value=bdg),
               gr.update(value='<p style="color:#3fb950">✅ Analysis complete!</p>'),
               gr.update(visible=True), gr.update(value=current),
               gr.update(visible=True), gr.update(visible=True),
               gr.update(visible=False), final, [], [])

    upload.change(
        fn=pipeline,
        inputs=[upload, role],
        outputs=[orig_img, heat_img, badge_html, status_html,
                 del_btn, expl_box, chat_btn, explain_grp,
                 chat_grp, state, cli_hist, pat_hist]
    )

    # ── ROLE SWITCH ────────────────────────────────

    def switch_role(sel_role, data, c_hist, p_hist):
        # Issue 2 fix: show correct explanation for selected role
        report = ""
        if data:
            report = data.get("clinician_report","") if sel_role=="Clinician" \
                     else data.get("patient_report","")
        # Issue 3 fix: show correct chat history for selected role
        history = c_hist if sel_role=="Clinician" else p_hist
    
        return gr.update(value=report), gr.update(value=history)

    
    role.change(
    fn=switch_role,
    inputs=[role, state, cli_hist, pat_hist],
    outputs=[expl_box, chatbot]   # ← updates BOTH explanation AND chatbot
    )

    # ── START CHAT ─────────────────────────────────
    def start_chat(sel_role, data, c_hist, p_hist):
        if not data:
            return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), c_hist, p_hist, []
        report  = data.get("clinician_report","") if sel_role=="Clinician" else data.get("patient_report","")
        history = c_hist if sel_role=="Clinician" else p_hist
        if not history:
            history = [{"role":"assistant","content":report}]
        if sel_role=="Clinician": c_hist = history
        else: 
            p_hist = history
        return (gr.update(visible=False), gr.update(visible=True),
                gr.update(visible=False), c_hist, p_hist, history)

    chat_btn.click(
        fn=start_chat,
        inputs=[role, state, cli_hist, pat_hist],
        outputs=[explain_grp, chat_grp, chat_btn, cli_hist, pat_hist, chatbot])

    # ── SEND MESSAGE ───────────────────────────────
    def send_msg(msg, sel_role, data, c_hist, p_hist):
        if not msg.strip() or not data:
            return "", (c_hist if sel_role=="Clinician" else p_hist), c_hist, p_hist
        history  = list(c_hist if sel_role=="Clinician" else p_hist)
        response = get_chat_resp(msg.strip(), history, sel_role, data)
        history.append({"role":"user","content":msg.strip()})
        history.append({"role":"assistant","content":response})
        if sel_role=="Clinician": c_hist = history
        else:                     p_hist = history
        return "", history, c_hist, p_hist

    send_btn.click(fn=send_msg,
        inputs=[chat_in, role, state, cli_hist, pat_hist],
        outputs=[chat_in, chatbot, cli_hist, pat_hist])
    chat_in.submit(fn=send_msg,
        inputs=[chat_in, role, state, cli_hist, pat_hist],
        outputs=[chat_in, chatbot, cli_hist, pat_hist])

    # ── BACK / CLEAR / DELETE ──────────────────────
    back_btn.click(
        fn=lambda r,d: (gr.update(visible=True), gr.update(visible=False), gr.update(visible=True),
                        gr.update(value=d.get("clinician_report","") if r=="Clinician" else d.get("patient_report",""))),
        inputs=[role, state],
        outputs=[explain_grp, chat_grp, chat_btn, expl_box]
    )
    clear_btn.click(
        fn=lambda r,d,c,p: ([{"role":"assistant","content":d.get("clinician_report" if r=="Clinician" else "patient_report","")}],
                             [{"role":"assistant","content":d.get("clinician_report","")}] if r=="Clinician" else c,
                             [{"role":"assistant","content":d.get("patient_report","")}]   if r=="Patient"   else p),
        inputs=[role, state, cli_hist, pat_hist],
        outputs=[chatbot, cli_hist, pat_hist]
    )
    del_btn.click(
        fn=lambda: (None,None,None,gr.update(value=""),gr.update(value=""),
                    gr.update(visible=False),gr.update(value="",placeholder="Upload a chest X-ray to begin…"),
                    gr.update(visible=False),gr.update(visible=True),gr.update(visible=False),{},[],[],[]),
        outputs=[upload,orig_img,heat_img,badge_html,status_html,del_btn,
                 expl_box,chat_btn,explain_grp,chat_grp,state,cli_hist,pat_hist,chatbot]
    )

# ════════════════════════════════════════════════════
# LAUNCH — share=True gives public URL, NO tunnel needed
# ════════════════════════════════════════════════════
demo.queue(max_size=2)
demo.launch(
    share=True,          # ← automatic public URL, no cloudflared needed
    server_port=7861,
    debug=False,
    quiet=False

)


/tmp/ipykernel_55/785029225.py:259: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="CheXplain") as demo:
/tmp/ipykernel_55/785029225.py:296: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="", height=360, type="messages", allow_tags=False)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://1f0d6172069eeaab15.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 Hooks registered on: backbone.features.denseblock4


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
2026-03-06 06:50:36.882184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772779837.080714     210 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772779837.136770     210 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772779837.594750     210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772779837.594798     210 computation_placer.cc:177] computation placer already registered. Please check linkage 